# Late fusion and internal evaluation

**Scientific purpose.** Document fold-specific multinomial logistic-regression fusion,
verify the exact eight-feature full-fusion contract, and inspect released aggregate
internal performance and DeLong results.

**Runnability classification:** public aggregate reanalysis is runnable from a clean
clone. Refitting fusion or recalculating patient-level metrics requires private OOF,
held-out predictions, clinical variables, and fold assignments.

**Inputs:** released aggregate tables; private OOF/test tables for optional refitting.
**Outputs:** displayed aggregate checks; any patient-linked refit products remain private.

The full-fusion input is W4 probabilities, W5 probabilities, age, and sex only. W3 is a
standalone comparator and is excluded.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

from liver_tumor_pipeline.constants import CLASS_ORDER, FULL_FUSION_FEATURES
from liver_tumor_pipeline.fusion import build_full_fusion_features

expected_order = (
    "w4_p_HCC", "w4_p_ICC", "w4_p_CHCC",
    "w5_p_HCC", "w5_p_ICC", "w5_p_CHCC",
    "age", "sex",
)
if FULL_FUSION_FEATURES != expected_order or any(name.startswith("w3_") for name in expected_order):
    raise RuntimeError("Eight-feature full-fusion contract is inconsistent.")
pd.DataFrame({"position": range(1, 9), "feature": expected_order})


## Fold-specific fusion definition

Each outer fold fits median imputation, standardization, and balanced multinomial
logistic regression using the other four OOF folds, then predicts its held-out fold.
Five fold models are ensembled for the independent 56-patient evaluation cohort.


In [ ]:
def fit_fusion_fold(train_frame, validation_frame, test_frame):
    feature_names = list(FULL_FUSION_FEATURES)
    X_train = train_frame[feature_names].to_numpy(dtype=np.float32)
    X_validation = validation_frame[feature_names].to_numpy(dtype=np.float32)
    X_test = test_frame[feature_names].to_numpy(dtype=np.float32)
    y_train = train_frame["label"].to_numpy()

    for matrix in (X_train, X_validation, X_test):
        matrix[matrix <= -1] = np.nan
    imputer = SimpleImputer(strategy="median").fit(X_train)
    X_train = imputer.transform(X_train)
    X_validation = imputer.transform(X_validation)
    X_test = imputer.transform(X_test)
    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train)
    X_validation = scaler.transform(X_validation)
    X_test = scaler.transform(X_test)
    model = LogisticRegression(
        max_iter=2000,
        C=1.0,
        class_weight="balanced",
        random_state=42,
    )
    model.fit(X_train, y_train)
    return {
        "model": model,
        "imputer": imputer,
        "scaler": scaler,
        "validation_probabilities": model.predict_proba(X_validation),
        "test_probabilities": model.predict_proba(X_test),
    }


def macro_metrics(labels, probabilities):
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = probabilities.argmax(axis=1)
    return {
        "macro_auc": roc_auc_score(labels, probabilities, multi_class="ovr", average="macro"),
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
    }


## Released aggregate results

These tables contain no patient-level rows. Held-out internal macro one-versus-rest AUC
is the principal evidence because validation-selected CNN OOF predictions and
development-wide CNN class weights can make CV/stacking estimates optimistic.


In [ ]:
performance_path = REPO_ROOT / "results" / "aggregate" / "internal_performance.csv"
delong_path = REPO_ROOT / "results" / "aggregate" / "delong_summary.csv"
for required in (performance_path, delong_path):
    if not required.is_file():
        raise FileNotFoundError(f"Released aggregate artifact is unavailable: {required.name}")

internal_performance = pd.read_csv(performance_path)
delong_summary = pd.read_csv(delong_path)
internal_performance


## Statistical interpretation

Nine exploratory per-class two-sided DeLong comparisons were evaluated. The smallest
raw p-value was 0.0293 for full fusion versus W4 in cHCC-CCA, not full fusion versus W5.
Full fusion versus W5 had p=0.3457. No comparison remained significant after either
Bonferroni or Benjamini-Hochberg adjustment across the nine comparisons.

Patient-level predictions are intentionally absent from the repository; therefore this
notebook inspects the released aggregate statistical table rather than recreating tests
from undistributed probabilities.


In [ ]:
required_delong_columns = {
    "comparison", "class", "reported_raw_p", "uncorrected_p_recomputed",
    "bonferroni_p_9", "bh_p_9"
}
missing_delong_columns = sorted(required_delong_columns.difference(delong_summary.columns))
if missing_delong_columns:
    raise ValueError(f"DeLong aggregate schema is incomplete: {missing_delong_columns}")
delong_summary.sort_values("reported_raw_p").reset_index(drop=True)
